# Getting to Know the Data

## Research Question
> Are property prices in Mexico more influenced by property size or by location?

## What This Notebook Does
This notebook covers the first step of the analysis pipeline:
- Loading three raw CSV files
- Inspecting each file for data quality issues
- Cleaning each file using method chaining
- Combining them into one master dataset
- Producing first descriptive statistics

## 1. Setup

In [1]:
import pandas as pd

print("pandas version:", pd.__version__)

pandas version: 2.3.3


## 2. Loading and Inspecting the Raw Data

Before cleaning anything we inspect each file to understand 
its structure and identify problems.
We use 5 methods on every new dataset:
- `.head()` — see the first few rows
- `.shape` — number of rows and columns
- `.info()` — column types and null counts
- `.dtypes` — data type of each column
- `.isnull().sum()` — count of missing values per column

In [2]:
path_1 = "../data/mexico-real-estate-1.csv"
path_2 = "../data/mexico-real-estate-2.csv"
path_3 = "../data/mexico-real-estate-3.csv"

# Inspect file 1
df_raw1 = pd.read_csv(path_1)
print("=== File 1 ===")
print("Shape:", df_raw1.shape)
print("\nMissing values:")
print(df_raw1.isnull().sum())
df_raw1.head()

=== File 1 ===
Shape: (700, 6)

Missing values:
property_type      0
state              0
lat              117
lon              117
area_m2            0
price_usd          0
dtype: int64


,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.560181,-99.233528,150.0,"$67,965.56"
1,house,Nuevo León,25.688436,-100.198807,186.0,"$63,223.78"
2,apartment,Guerrero,16.767704,-99.764383,82.0,"$84,298.37"
3,apartment,Guerrero,16.829782,-99.911012,150.0,"$94,308.80"
4,house,Veracruz de Ignacio de la Llave,NaN,NaN,175.0,"$94,835.67"


## 3. Cleaning Each Dataset

Each file has different problems that need fixing before combining.

**File 1:** `price_usd` is stored as a string with `$` and `,` symbols — needs converting to float  
**File 2:** Price is in Mexican Pesos — needs converting to USD (÷ 19)  
**File 3:** lat and lon are combined in one column — needs splitting. State name is buried inside a pipe-separated string — needs extracting



In [3]:
# File 1 Cleaning
df1 = (
    pd.read_csv(path_1)
    .dropna()
    .assign(
        price_usd=lambda x: x["price_usd"]
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .astype(float)
    )
)

print("df1 shape:", df1.shape)
df1.head()

df1 shape: (583, 6)


,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.560181,-99.233528,150.0,67965.56
1,house,Nuevo León,25.688436,-100.198807,186.0,63223.78
2,apartment,Guerrero,16.767704,-99.764383,82.0,84298.37
3,apartment,Guerrero,16.829782,-99.911012,150.0,94308.80
5,house,Yucatán,21.052583,-89.538639,205.0,105191.37


In [4]:
# Clean File 2
df2 = (
    pd.read_csv(path_2)
    .assign(price_usd=lambda x: x["price_mxn"].div(19))
    .drop(columns=["price_mxn"])
    .dropna()
)

print("df2 shape:", df2.shape)
df2.head()

df2 shape: (571, 6)


,property_type,state,lat,lon,area_m2,price_usd
0,apartment,Nuevo León,25.721081,-100.345581,72.0,68421.052632
2,house,Morelos,23.634501,-102.552788,360.0,278947.368421
6,apartment,Estado de México,19.272040,-99.572013,85.0,65789.473684
7,house,San Luis Potosí,22.138882,-100.996510,158.0,111578.947368
8,apartment,Distrito Federal,19.394558,-99.129707,65.0,39904.736842


In [5]:
# Clean File 3
df3 = (
    pd.read_csv(path_3)
    .dropna()
    .assign(
        lat=lambda x: x["lat-lon"]
            .str.split(",", expand=True)[0]
            .astype(float),
        lon=lambda x: x["lat-lon"]
            .str.split(",", expand=True)[1]
            .astype(float),
        state=lambda x: x["place_with_parent_names"]
            .str.split("|", expand=True)[2]
    )
    .drop(columns=["place_with_parent_names", "lat-lon"])
    .dropna()
)

print("df3 shape:", df3.shape)
df3.head()

df3 shape: (582, 6)


,property_type,area_m2,price_usd,lat,lon,state
0,apartment,71.0,48550.59,19.525890,-99.151703,Distrito Federal
1,house,233.0,168636.73,19.264054,-99.572753,Estado de México
2,house,300.0,86932.69,19.268629,-99.671722,Estado de México
4,apartment,84.0,68508.67,19.511938,-96.871956,Veracruz de Ignacio de la Llave
5,house,175.0,102763.00,20.689157,-103.366728,Jalisco


## 4. Combining Into One Master Dataset

Now that all three files share the same column structure 
we stack them vertically using `pd.concat()`.
`ignore_index=True` resets the row numbers so they run 
continuously from 0 instead of restarting for each file.

In [6]:
# Verify all three have matching columns before combining
print("Columns match:", set(df1.columns) == set(df2.columns) == set(df3.columns))

# Combine
df = pd.concat([df1, df2, df3], ignore_index=True)
print(f"\nCombined shape: {df.shape}")
df.head()

Columns match: True

Combined shape: (1736, 6)


,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.560181,-99.233528,150.0,67965.56
1,house,Nuevo León,25.688436,-100.198807,186.0,63223.78
2,apartment,Guerrero,16.767704,-99.764383,82.0,84298.37
3,apartment,Guerrero,16.829782,-99.911012,150.0,94308.80
4,house,Yucatán,21.052583,-89.538639,205.0,105191.37


## 5. First Look at the Numbers

In [7]:
# Summary statistics
pd.options.display.float_format = "{:,.2f}".format
df[["price_usd", "area_m2"]].describe()

,price_usd,area_m2
count,"1,736.00","1,736.00"
mean,"115,331.98",170.26
std,"65,426.17",80.59
min,"33,157.89",60.00
25%,"65,789.47",101.75
50%,"99,262.13",156.00
75%,"150,846.66",220.00
max,"326,733.66",385.00


## 6. Save the Clean Dataset

In [8]:
df.to_csv("../data/mexico-real-estate-combined-clean.csv", index=False)
print("Saved successfully!")
print(f"Final dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Saved successfully!
Final dataset: 1736 rows, 6 columns


## Key Findings from This Notebook

- Combined dataset has **1,736 property listings** across Mexico
- Two property types: **houses** and **apartments**
- Price ranges from roughly **$10,000 to $320,000 USD**
- Mean price (~$115,000) is higher than median (~$97,000) 
  — confirming **right skew** from luxury properties
- Area ranges from small apartments to large estates 
  — also right skewed
- Data covers multiple Mexican states including 
  Distrito Federal, Estado de México, and Yucatán